# release16_post_short_training_reward_checkpoint_contract_v1

Post-short-training reward/checkpoint contract for the release16 retention path.

This notebook starts from the failed `release16_drawdown_drift_short_training_screen_v1` and the newer constrained checkpoint repair screen. It does not promote an actor. It decides whether the next step is another PPO run, a reward/checkpoint objective revision, or a target/BC slice repair before training.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src" / "soxl_quant").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.reward_checkpoint_screen import (
    RewardCheckpointScreenConfig,
    render_reward_checkpoint_screen_review,
    score_reward_checkpoint_candidates,
    summarize_reward_checkpoint_screen,
)

EXPERIMENT = "release16_post_short_training_reward_checkpoint_contract_v1"
OUT = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE_REPAIR = ROOT / "runtime" / "model_selection" / "card" / "release16_constrained_checkpoint_repair_screen_v1"
SOURCE_SHORT = ROOT / "runtime" / "model_selection" / "card" / "release16_drawdown_drift_short_training_screen_v1"
OUT.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 20.0
REQUIRED_WINDOWS = ["wf_2020", "wf_2022", "wf_2023_2024"]
SLICE_ACCURACY_FLOOR = 0.60

print(f"root={ROOT}")
print(f"out={OUT.relative_to(ROOT)}")


root=/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
out=runtime/model_selection/card/release16_post_short_training_reward_checkpoint_contract_v1


## Governance

- Baselines: canonical `soxlbacktest_qlib_walkforward_baseline_v1`; direct comparison is the latest release16 constrained repair screen.
- Role: `diagnose`; this is a training-contract gate, not promotion.
- Changed layer: `reward` and `selection`.
- Acceptance criteria: every required walk-forward window must reach relative return `>= +30pp` and MaxDD `< 20%`; actor retention remains disabled until official three-fold replay proves the gate.
- Historical success/failure reference: the closest prior success is the deterministic SOXLBacktest/Qlib baseline for stable reruns; the closest failure is `release16_drawdown_drift_short_training_screen_v1`, where PPO drift did not clear any fold. This run is not a duplicate because it explicitly checks whether the post-short-training blocker is reward/checkpoint objective or BC holdout slice readiness.

In [2]:
repair_summary = json.loads((SOURCE_REPAIR / "summary.json").read_text(encoding="utf-8"))
short_summary = json.loads((SOURCE_SHORT / "summary.json").read_text(encoding="utf-8"))
metrics = pd.read_csv(SOURCE_SHORT / "checkpoint_candidate_metrics.csv")
selection_by_fold = pd.read_csv(SOURCE_SHORT / "checkpoint_selection_by_fold.csv")
bc_slice = pd.read_csv(SOURCE_SHORT / "bc_slice_metrics.csv")

print({
    "repair_decision": repair_summary.get("decision"),
    "repair_next": repair_summary.get("next_mechanism"),
    "short_decision": short_summary.get("decision"),
    "short_next": short_summary.get("next_mechanism"),
    "candidate_rows": len(metrics),
    "bc_slice_rows": len(bc_slice),
})


{'repair_decision': 'constrained_checkpoint_selector_no_promotion', 'repair_next': 'train_reward_constrained_short_training_candidate_with_relative_floor_and_drawdown_gap_loss_then_official_three_fold_replay', 'short_decision': 'release16_drawdown_drift_short_training_screen_mixed_no_promotion', 'short_next': 'revise_reward_patch_or_checkpoint_selection_before_full_training', 'candidate_rows': 12, 'bc_slice_rows': 24}


In [3]:
def classify_failure(relative_gap: float, max_dd_gap: float) -> str:
    if relative_gap <= 0.0 and max_dd_gap <= 0.0:
        return "pass"
    if relative_gap > 0.0 and max_dd_gap > 0.0:
        return "joint_tradeoff"
    if relative_gap > 0.0:
        return "underparticipation"
    return "drawdown_control"

rows = []
for row in metrics.to_dict("records"):
    relative_return = float(row["test_relative_return"])
    max_dd = float(row["test_max_dd"])
    relative_gap = max(0.0, TARGET_RELATIVE_RETURN_PP - relative_return)
    max_dd_gap = max(0.0, max_dd - TARGET_MAX_DD_PCT)
    gate_pass = relative_gap <= 0.0 and max_dd_gap <= 0.0
    rows.append(
        {
            "profile": str(row["candidate"]),
            "window": str(row["fold"]),
            "relative_return": relative_return,
            "max_dd": max_dd,
            "avg_pos": float(row.get("test_avg_pos", 0.0)),
            "trade_count": int(float(row.get("test_trade_count", 0) or 0)),
            "relative_gap": relative_gap,
            "max_dd_gap": max_dd_gap,
            "gate_pass": bool(gate_pass),
            "failure_type": classify_failure(relative_gap, max_dd_gap),
            "daily_path": str(row.get("daily_path", "")),
        }
    )

gate_gaps = pd.DataFrame(rows).sort_values(["profile", "window"]).reset_index(drop=True)
gate_gaps.to_csv(OUT / "gate_gaps.csv", index=False)

gate_gaps.groupby("profile", sort=True).agg(
    gate_pass_count=("gate_pass", "sum"),
    window_count=("window", "count"),
    min_relative_return=("relative_return", "min"),
    max_max_dd=("max_dd", "max"),
).reset_index()


,profile,gate_pass_count,window_count,min_relative_return,max_max_dd
0,bc_initialization,0,3,-264.728894,26.832912
1,ppo_retained_iter_0_bc_equivalent,0,3,-264.728894,26.832912
2,ppo_retained_iter_4,0,3,-262.992258,37.397618
3,ppo_selected_best,0,3,-262.992258,37.397618


In [4]:
scored = score_reward_checkpoint_candidates(
    gate_gaps,
    reference_candidate="bc_initialization",
    config=RewardCheckpointScreenConfig(
        target_relative_return=TARGET_RELATIVE_RETURN_PP,
        target_max_dd=TARGET_MAX_DD_PCT,
        max_dd_gap_weight=10.0,
    ),
)
reward_summary = summarize_reward_checkpoint_screen(scored)
review = render_reward_checkpoint_screen_review(scored, reward_summary)

scored.to_csv(OUT / "reward_checkpoint_scores.csv", index=False)
(OUT / "reward_checkpoint_review.md").write_text(review, encoding="utf-8")

print(reward_summary)
scored[[
    "candidate",
    "full_gate",
    "reward_checkpoint_loss",
    "gate_pass_count",
    "window_count",
    "min_relative_return",
    "max_max_dd",
    "failure_mix",
    "required_objective_terms",
]]


{'decision': 'reward_checkpoint_screen_no_promotion', 'selected_candidate': 'bc_initialization', 'selected_reward_checkpoint_loss': 401.48433619394115, 'selected_gate_gap_loss': 401.48433619394115, 'selected_failure_mix': 'drawdown_control:2,joint_tradeoff:1', 'selected_min_relative_return': -264.72889421266575, 'selected_max_max_dd': 26.8329120183762, 'promotion_allowed': False, 'ppo_allowed_next': False, 'required_objective_terms': ['checkpoint_objective_loss', 'drawdown_drift_penalty', 'trend_preservation_floor'], 'next_mechanism': 'train_drawdown_drift_reward_or_checkpoint_recipe'}


,candidate,full_gate,reward_checkpoint_loss,gate_pass_count,window_count,min_relative_return,max_max_dd,failure_mix,required_objective_terms
0,bc_initialization,False,401.484336,0,3,-264.728894,26.832912,"drawdown_control:2,joint_tradeoff:1","[""checkpoint_objective_loss"", ""drawdown_drift_..."
1,ppo_retained_iter_0_bc_equivalent,False,401.484336,0,3,-264.728894,26.832912,"drawdown_control:2,joint_tradeoff:1","[""checkpoint_objective_loss"", ""drawdown_drift_..."
2,ppo_retained_iter_4,False,668.201498,0,3,-262.992258,37.397618,"joint_tradeoff:2,drawdown_control:1","[""checkpoint_objective_loss"", ""drawdown_drift_..."
3,ppo_selected_best,False,668.201498,0,3,-262.992258,37.397618,"joint_tradeoff:2,drawdown_control:1","[""checkpoint_objective_loss"", ""drawdown_drift_..."


In [5]:
holdout = bc_slice[bc_slice["target_split"].astype(str).eq("holdout")].copy()
state_slices = {"release_gate_state", "is_downtrend_state", "drawdown_control_state"}
holdout = holdout[holdout["slice"].astype(str).isin(state_slices)].copy()
holdout["accuracy"] = pd.to_numeric(holdout["accuracy"], errors="coerce")
holdout["samples"] = pd.to_numeric(holdout["samples"], errors="coerce").fillna(0).astype(int)
holdout["slice_gate"] = holdout["accuracy"].ge(SLICE_ACCURACY_FLOOR)
holdout["failure_reason"] = holdout.apply(
    lambda row: "pass" if bool(row["slice_gate"]) else f"accuracy_below_{SLICE_ACCURACY_FLOOR:.2f}",
    axis=1,
)
holdout_readout = holdout[["fold", "slice", "samples", "accuracy", "slice_gate", "failure_reason"]].sort_values(["fold", "slice"])
holdout_readout.to_csv(OUT / "bc_holdout_slice_readout.csv", index=False)

selected_holdout_gate_by_fold = {
    str(row["fold"]): bool(row["selected_holdout_slice_gate"])
    for row in selection_by_fold.to_dict("records")
}
bc_holdout_slice_gate = bool(holdout_readout["slice_gate"].all()) and all(selected_holdout_gate_by_fold.values())

print({
    "bc_holdout_slice_gate": bc_holdout_slice_gate,
    "selected_holdout_gate_by_fold": selected_holdout_gate_by_fold,
    "failed_slice_rows": int((~holdout_readout["slice_gate"]).sum()),
})
holdout_readout


{'bc_holdout_slice_gate': False, 'selected_holdout_gate_by_fold': {'wf_2020': False, 'wf_2022': False, 'wf_2023_2024': False}, 'failed_slice_rows': 7}


,fold,slice,samples,accuracy,slice_gate,failure_reason
5,wf_2020,drawdown_control_state,158,0.626582,True,pass
6,wf_2020,is_downtrend_state,157,0.630573,True,pass
7,wf_2020,release_gate_state,13,0.461538,False,accuracy_below_0.60
13,wf_2022,drawdown_control_state,6,0.333333,False,accuracy_below_0.60
14,wf_2022,is_downtrend_state,24,0.541667,False,accuracy_below_0.60
15,wf_2022,release_gate_state,3,0.000000,False,accuracy_below_0.60
21,wf_2023_2024,drawdown_control_state,7,0.285714,False,accuracy_below_0.60
22,wf_2023_2024,is_downtrend_state,98,0.510204,False,accuracy_below_0.60
23,wf_2023_2024,release_gate_state,3,0.333333,False,accuracy_below_0.60


In [6]:
selected_candidate = str(reward_summary.get("selected_candidate") or "")
selected_gate = gate_gaps[gate_gaps["profile"].eq(selected_candidate)].copy()
selected_by_fold = {}
for row in selected_gate.to_dict("records"):
    selected_by_fold[str(row["window"])] = {
        "selected_candidate": selected_candidate,
        "selected_relative_return": float(row["relative_return"]),
        "selected_max_dd": float(row["max_dd"]),
        "selected_avg_pos": float(row["avg_pos"]),
        "selected_trade_count": int(row["trade_count"]),
        "selected_candidate_gate": bool(row["gate_pass"]),
        "failure_type": str(row["failure_type"]),
        "relative_gap_to_target_pp": float(row["relative_gap"]),
        "max_dd_gap_to_target_pp": float(row["max_dd_gap"]),
        "selected_daily_path": str(row["daily_path"]),
    }

failed_folds = [fold for fold, payload in selected_by_fold.items() if not payload["selected_candidate_gate"]]
required_terms = list(reward_summary.get("required_objective_terms", []))
target_contract_gate = set(required_terms) >= {"checkpoint_objective_loss", "drawdown_drift_penalty", "trend_preservation_floor"}
training_allowed_next = bool(target_contract_gate and bc_holdout_slice_gate and not failed_folds)
# No candidate passes the official gate here, so training remains a target/BC repair task rather than actor continuation.
training_allowed_next = False
next_action = "rebuild_fold_local_joint_target_with_holdout_slice_gate_then_reward_checkpoint_screen"

contract = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_experiment": "release16_constrained_checkpoint_repair_screen_v1",
    "source_short_training_experiment": "release16_drawdown_drift_short_training_screen_v1",
    "decision": "post_short_training_reward_checkpoint_contract_requires_bc_slice_repair",
    "selected_candidate": selected_candidate,
    "target_relative_return_pp": TARGET_RELATIVE_RETURN_PP,
    "target_max_dd_pct": TARGET_MAX_DD_PCT,
    "required_wf_windows": REQUIRED_WINDOWS,
    "required_objective_terms": required_terms,
    "target_contract_gate": bool(target_contract_gate),
    "bc_holdout_slice_gate": bool(bc_holdout_slice_gate),
    "selected_holdout_gate_by_fold": selected_holdout_gate_by_fold,
    "training_allowed_next": bool(training_allowed_next),
    "ppo_allowed_next": False,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "next_action": next_action,
}

summary = {
    **contract,
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": "release16_constrained_checkpoint_repair_screen_v1",
    "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "role": "diagnose",
    "changed_layer": "reward,selection",
    "decision": "release16_post_short_training_contract_no_training",
    "primary_blocker": "bc_candidate_or_slice_gate_failed",
    "next_mechanism": next_action,
    "next_training_allowed_after_target_revision": False,
    "selected_reward_checkpoint_loss": float(reward_summary.get("selected_reward_checkpoint_loss", 0.0)),
    "selected_gate_gap_loss": float(reward_summary.get("selected_gate_gap_loss", 0.0)),
    "selected_failure_mix": str(reward_summary.get("selected_failure_mix", "")),
    "selected_min_relative_return": float(reward_summary.get("selected_min_relative_return", 0.0)),
    "selected_max_max_dd": float(reward_summary.get("selected_max_max_dd", 0.0)),
    "worst_wf_max_drawdown_pct": float(max(payload["selected_max_dd"] for payload in selected_by_fold.values())),
    "min_wf_relative_return_pp": float(min(payload["selected_relative_return"] for payload in selected_by_fold.values())),
    "max_drawdown_gap_to_target_pp": float(max(payload["max_dd_gap_to_target_pp"] for payload in selected_by_fold.values())),
    "max_relative_gap_to_target_pp": float(max(payload["relative_gap_to_target_pp"] for payload in selected_by_fold.values())),
    "failed_folds": failed_folds,
    "selected_by_fold": selected_by_fold,
    "sidecars": [
        "gate_gaps.csv",
        "reward_checkpoint_scores.csv",
        "bc_holdout_slice_readout.csv",
        "reward_checkpoint_review.md",
        "training_contract.json",
        "summary.json",
        "conclusion.md",
    ],
}

(OUT / "training_contract.json").write_text(json.dumps(contract, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({k: summary[k] for k in ["decision", "primary_blocker", "selected_candidate", "training_allowed_next", "bc_holdout_slice_gate", "next_mechanism"]}, indent=2, sort_keys=True))


{
  "bc_holdout_slice_gate": false,
  "decision": "release16_post_short_training_contract_no_training",
  "next_mechanism": "rebuild_fold_local_joint_target_with_holdout_slice_gate_then_reward_checkpoint_screen",
  "primary_blocker": "bc_candidate_or_slice_gate_failed",
  "selected_candidate": "bc_initialization",
  "training_allowed_next": false
}


In [7]:
conclusion = f"""# Release16 Post-Short-Training Reward/Checkpoint Contract

## Decision

- decision: `{summary['decision']}`
- selected candidate: `{summary['selected_candidate']}`
- promotion_allowed: `{summary['promotion_allowed']}`
- actor_retention_allowed: `{summary['actor_retention_allowed']}`
- training_allowed_next: `{summary['training_allowed_next']}`
- primary blocker: `{summary['primary_blocker']}`

## Gate Readout

- Required gate: relative return >= `{TARGET_RELATIVE_RETURN_PP:.2f}` pp and MaxDD < `{TARGET_MAX_DD_PCT:.2f}%` on `{', '.join(REQUIRED_WINDOWS)}`.
- Selected candidate min relative return: `{summary['selected_min_relative_return']:.2f}` pp.
- Selected candidate worst MaxDD: `{summary['selected_max_max_dd']:.2f}%`.
- Failed folds: `{', '.join(summary['failed_folds'])}`.

## Objective Terms

"""
for term in summary["required_objective_terms"]:
    conclusion += f"- `{term}`\n"
conclusion += f"""
## BC Slice Blocker

The reward/checkpoint objective terms are expressible in the current reward/config code, but the short-training BC holdout slice gate is still false on the selected folds. Re-running longer PPO from the same weak BC target is not allowed by the review rules.

Next action: `{summary['next_mechanism']}`.

Artifacts: `gate_gaps.csv`, `reward_checkpoint_scores.csv`, `bc_holdout_slice_readout.csv`, `reward_checkpoint_review.md`, `training_contract.json`, `summary.json`.
"""
(OUT / "conclusion.md").write_text(conclusion, encoding="utf-8")
print(conclusion)


# Release16 Post-Short-Training Reward/Checkpoint Contract

## Decision

- decision: `release16_post_short_training_contract_no_training`
- selected candidate: `bc_initialization`
- promotion_allowed: `False`
- actor_retention_allowed: `False`
- training_allowed_next: `False`
- primary blocker: `bc_candidate_or_slice_gate_failed`

## Gate Readout

- Required gate: relative return >= `30.00` pp and MaxDD < `20.00%` on `wf_2020, wf_2022, wf_2023_2024`.
- Selected candidate min relative return: `-264.73` pp.
- Selected candidate worst MaxDD: `26.83%`.
- Failed folds: `wf_2020, wf_2022, wf_2023_2024`.

## Objective Terms

- `checkpoint_objective_loss`
- `drawdown_drift_penalty`
- `trend_preservation_floor`

## BC Slice Blocker

The reward/checkpoint objective terms are expressible in the current reward/config code, but the short-training BC holdout slice gate is still false on the selected folds. Re-running longer PPO from the same weak BC target is not allowed by the review rules.

Next a

## Notebook Conclusion

This diagnostic does not retain an actor. The reward/checkpoint loss still selects `bc_initialization`, with all three required windows failing the `+30pp / <20% MaxDD` gate. The necessary objective terms are available (`checkpoint_objective_loss`, `drawdown_drift_penalty`, and `trend_preservation_floor`), but the BC holdout slice gate is false, so the next valid step is a fold-local joint target rebuild with a holdout slice gate before any new short PPO/training continuation.